# Antarctic lateral drag — supporting diagnostics

Companion to `LD-pub-workspace.ipynb`. This notebook owns the common-support FIP comparison, mechanics and thickness, actual lateral-drag face stresses, activation/retention, geometric localisation and pack checks. It uses the same configuration, experiments, periods and export directory. Run the main notebook first for the experiment manifest and FIA tables.

The old performance and paper2 notebooks mixed 1993 pilot runs, legacy APIs, estimated stress proxies, thickness exploration and wave forcing. Relevant publication diagnostics are retained below; pilot and out-of-scope wave recipes remain recoverable in Git history. Neither old results nor unverified case aliases are used as current evidence. All numerical stages default off until their input definitions are reviewed.

## 0. Shared configuration

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shlex
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = next((p for p in [Path.cwd(), Path.cwd()/'notebooks',
    Path(os.environ.get('CICE_FREE_SLIP_REPO', '.'))/'notebooks']
    if (p/'ld_pub_workflow.py').is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Start in the repository/notebooks directory, or set CICE_FREE_SLIP_REPO')
sys.path.insert(0, str(NOTEBOOK_DIR))
import ld_pub_workflow as wf
SHUGA_REPO = os.environ.get('SHUGA_REPO')
if SHUGA_REPO:
    shuga_repo = Path(SHUGA_REPO).expanduser()
    sys.path.insert(0, str(shuga_repo.parent if (shuga_repo/'__init__.py').exists() else shuga_repo))
import shuga
from shuga.core.types import RunSpec, ClassificationSpec, MetricsSpec
from shuga.core.paths import ShugaPaths
from shuga.core.regions import ANTARCTIC_8_REGIONS

CONFIG_PATH = Path(os.environ.get('LD_PUB_CONFIG', NOTEBOOK_DIR/'ld_pub_config.json'))
CFG = json.loads(CONFIG_PATH.read_text())
START, END = CFG['model_start'], CFG['model_end']
OBS_START, OBS_END = CFG['evaluation_start'], CFG['evaluation_end']
GROUP, METHOD = CFG['experiment_group'], CFG['method']
BANK = Path(CFG['afim_output_root'])
ARCHIVE = BANK
EXPORT = Path(os.environ.get('LD_PUB_EXPORT', str(Path(CFG['graphics_root'])/GROUP/'publication')))
EXPORT.mkdir(parents=True, exist_ok=True)
CACHE = EXPORT/'cache'; CACHE.mkdir(exist_ok=True)
manifest = wf.expected_experiments()
EXPERIMENTS = manifest.experiment.tolist()
FAMILIES = {family: part.experiment.tolist() for family, part in manifest.groupby('family', sort=False)}
REFERENCE = 'no-slip-LFI-match'
PRIMARY = ['no-slip-LFI-match','free-slip-no-drag','Cs-high','Cq-high']
TENSILE_CASES = ['Cs-high-ktens-zero','Cs-high-ktens-low','Cs-high-ktens-mid','Cs-high']
DIAGNOSTIC_CASES = list(dict.fromkeys(PRIMARY+TENSILE_CASES))
PAIRS = {'boundary_only':('free-slip-no-drag','no-slip-LFI-match'),
         'static_drag':('Cs-high','free-slip-no-drag'),
         'quadratic_drag':('Cq-high','free-slip-no-drag'),
         'tensile_term':('Cs-high','Cs-high-ktens-zero'),
         'coast_increment_over_gib':('Cs-high','GIB-drag'),
         'gib_increment_over_coast':('Cs-high','cst-drag')}
REGIONS = list(ANTARCTIC_8_REGIONS)
OBS_FIA_PATH = Path(CFG['obs_fia_path'])
UNIT_OVERRIDES = {}  # Only documented metadata corrections; never inferred from values.
OPEN, COVERAGE, ARTIFACTS = {}, [], []

def contexts(name, method=METHOD, ice_type='FI'):
    run=RunSpec(name,START,END,project=CFG['project'],user=CFG['user'],experiment_group=GROUP)
    cls=ClassificationSpec(ice_type=ice_type,methods=method,grid_type=CFG['grid_type'],
        ispd_thresh=CFG['ispd_thresh'],aice_thresh=CFG['aice_thresh'],
        bin_window=CFG['bin_window'],bin_min_days=CFG['bin_min_days'],roll_window=CFG['roll_window'])
    met=MetricsSpec(methods=method)
    paths=ShugaPaths(run_cfg=run,cls_cfg=cls,met_cfg=met,afim_output_root=BANK,
                     archive_root=ARCHIVE,graphics_root=CFG['graphics_root'])
    return dict(run_cfg=run,cls_cfg=cls,met_cfg=met,pth_cfg=paths)

def load(name,kind,variables,method=METHOD,ice_type='FI'):
    key=(name,kind,tuple(variables),method,ice_type)
    if key not in OPEN:
        kwargs=contexts(name,method,ice_type)
        if kind!='history': kwargs['classification']=method
        loader={'metrics':shuga.load_metrics,'mask':shuga.load_classified,'history':shuga.load_cice}[kind]
        ds=loader(**kwargs,variables=variables,dt0_str=START,dtN_str=END)
        missing=set(variables)-set(ds.variables)
        if missing: ds.close(); raise KeyError(f'{name}: missing {missing}')
        OPEN[key]=ds
    return OPEN[key]

def open_product(path):
    path=Path(path)
    return xr.open_zarr(path,consolidated=False) if path.suffix=='.zarr' else xr.open_dataset(path)

def audited(da,name):
    da=da.copy(deep=False)
    if (name,da.name) in UNIT_OVERRIDES:
        da.attrs=dict(da.attrs,units=UNIT_OVERRIDES[(name,da.name)])
    return da

def git_commit(path):
    try:return subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):return 'UNVERIFIED'

provenance=dict(config=CFG,helper_sha256=wf.file_sha256(NOTEBOOK_DIR/'ld_pub_workflow.py'),
    model_repository_commit=git_commit(NOTEBOOK_DIR.parent),shuga_module=shuga.__file__,
    shuga_commit=git_commit(Path(shuga.__file__).parent),unit_overrides={str(k):v for k,v in UNIT_OVERRIDES.items()})
SIGNATURE=wf.signature(provenance)

def register(path,diagnostic,cases,period=None,notes=''):
    period=period or (OBS_START,OBS_END)
    ARTIFACTS.append(dict(path=str(path),diagnostic=diagnostic,experiments=list(cases),
        start=period[0],end=period[1],method=METHOD,signature=SIGNATURE,notes=notes))

def export_table(table,stem,cases,period=None):
    period=period or (OBS_START,OBS_END)
    path=EXPORT/f'{stem}_{period[0]}_{period[1]}.csv'
    table.to_csv(path,index=False);register(path,stem,cases,period)
    return path

print(f'Group: {GROUP}; model diagnostics: {START}–{END}; observational skill: {OBS_START}–{OBS_END}')
print('Inputs:',BANK/GROUP,'Outputs:',EXPORT)

In [ ]:
RUN_FIP=False
RUN_MECHANICS=False
RUN_STRESS=False
RUN_PACK=False
RUN_OCCURRENCE=False
RUN_ANCHOR=False
RUN_POINT_OBS=False
RUN_TENDENCIES=False
RECOMPUTE_MECHANICS=True
FIP_GRID_AND_SUPPORT_VERIFIED=False
HI_CONVENTION=None  # 'ice-covered-area' or 'grid-cell-volume-per-area'; inspect executed history definition.
LD_STRESS_DEFINITION_VERIFIED=False
NATIVE_GRID_VERIFIED=False
POINT_OBS_SAMPLING_VERIFIED=False
FORM_FACTOR_GRID_VERIFIED=False
MASK_TEMPORAL_SUPPORT_VERIFIED=False
NATIVE_AREA_PATH=None  # NetCDF: tarea [m²], TLON, TLAT on exact T grid; ocean_mask for point matching.
REGION_MASKS_PATH=None  # Common-grid sector masks, reviewed for gaps and overlap.
FIXED_PACK_MASK_PATH=None  # Native T-grid fixed_pack_mask, independent of experiment outcome.
FIP_INPUTS={}  # Experiment -> common-grid DAILY mod_mask, obs_mask, area [m²].
FORM_FACTOR_PATH=None  # Executed production file, variables FFx, FFy.
POINT_OBS_PATH=None  # Reviewed CSV: site,time,lon,lat,thickness_m; optional snow_m.
STRENGTH_VARIABLE='strength'
DIAGNOSTIC_REGION='Aus'  # Crop before expensive time reductions; repeat for retained sectors in PBS.
MAP_PAIRS={'boundary_only':PAIRS['boundary_only'],'static_drag':PAIRS['static_drag'],
           'quadratic_drag':PAIRS['quadratic_drag'],'tensile_term':PAIRS['tensile_term']}
RUN_PREPARE_FIP=False
OBS_DAILY_COMMON_PATH=None  # Reviewed obs_mask(time,y,x) and actual area(y,x), on EPSG:3031 x/y [m].
FIP_REMAP_VERIFIED=False
FIP_PIXEL_SIZE_M=5000
FIP_RADIUS_M=10000
CICE_LONGITUDE_SHIFT_DEG=0.0  # Change only with a documented coordinate justification.


## 1. FIP: common denominators, numerical skill, two-panel exports
The existing `FIP_differencing.py` averages model masks and observed FIC separately before regridding. Its maps are publication-ready only when the daily support is proven equivalent. Equal period titles alone do not establish this. When support differs, prepare matched **daily** masks on the common physical grid in the following optional section, then use this section. Verify longitude shift, projection, interpolation/mask semantics and radius of influence. Fractional regridded concentrations cannot silently be treated as binary occurrence.

`FIP_INPUTS` must have the schema stated in section 0, attribute `common_grid_verified=1`, and provenance for source stores, remapping, dates and validity. `REGION_MASKS_PATH` holds masks for quantitative sector areas, not plotting bounding boxes. A constant comparison-pixel area is allowed only if justified for the projection. The helpers retain true zero persistence and use per-pixel shared valid-day counts.

### Optional preparation on the reviewed observation grid

I'll first resolve whether each AF2020 composite represents an interval or a sample date and document any interval-to-daily representation. The following preparation accepts only that reviewed daily binary observation product; it does not invent observations between composites. `OBS_DAILY_COMMON_PATH` must include its support/missing-data policy in `temporal_support` and its source identity in `source`. I'll supply actual target cell areas and inspect physical alignment before setting the review flags.

Nearest-neighbour model remapping is performed one month at a time, before persistence averaging, using shuga's resampler. This can be expensive and belongs in an allocated analysis job. The output is retained under the publication directory, never over a source store. Existing products are not overwritten; remove/rebuild a specifically reviewed derived product when its provenance changes.

In [ ]:
if RUN_PREPARE_FIP:
    if not FIP_REMAP_VERIFIED or not FIP_GRID_AND_SUPPORT_VERIFIED or OBS_DAILY_COMMON_PATH is None or NATIVE_AREA_PATH is None:
        raise ValueError('Review observation support, target projection, native grid and remapping before preparation')
    from shuga.regridding.pyresample import area_definition_from_xy, resample_dataarray_to_area
    with open_product(NATIVE_AREA_PATH) as native:native=native.load()
    with open_product(OBS_DAILY_COMMON_PATH) as source:
        if not source.attrs.get('temporal_support') or not source.attrs.get('source'):
            raise ValueError('Record composite-to-daily policy and original observation source')
        if source.attrs.get('crs')!='EPSG:3031':raise ValueError('Verify target projection')
        observation=wf.binary_mask(source.obs_mask.sel(time=slice(OBS_START,OBS_END)))
        wf.daily_check(observation,OBS_START,OBS_END);wf.area_m2(source.area)
        area_def=area_definition_from_xy(source.x,source.y,pixel_size=FIP_PIXEL_SIZE_M)
        prepared=EXPORT/'prepared_fip';prepared.mkdir(exist_ok=True)
        for name in DIAGNOSTIC_CASES:
            path=prepared/f'{name}_{OBS_START}_{OBS_END}.zarr'
            if path.exists():raise FileExistsError(f'Review existing product before rebuilding: {path}')
            model=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask.sel(time=slice(OBS_START,OBS_END)))
            model,observation=wf.align_daily(model,observation,start=OBS_START,end=OBS_END)
            first=True
            for month in sorted(set(pd.DatetimeIndex(model.time.values).strftime('%Y-%m'))):
                part=model.sel(time=month)
                mapped=resample_dataarray_to_area(part,native.TLAT.values,
                    native.TLON.values+CICE_LONGITUDE_SHIFT_DEG,area_def,
                    radius=FIP_RADIUS_M,fill_value=np.nan,pixel_size=FIP_PIXEL_SIZE_M,name='mod_mask')
                observed=observation.sel(time=month)
                mapped,observed,area=xr.align(mapped,observed,source.area,join='exact')
                ds=xr.Dataset({'mod_mask':wf.binary_mask(mapped),'obs_mask':observed,'area':area})
                ds.attrs.update(common_grid_verified=1,analysis_signature=SIGNATURE,
                    obs_source=str(OBS_DAILY_COMMON_PATH),model_source=str(contexts(name)['pth_cfg'].resolve_cice_store()),
                    temporal_support=source.attrs['temporal_support'],remapping='nearest neighbour before persistence',
                    longitude_shift_deg=CICE_LONGITUDE_SHIFT_DEG,radius_m=FIP_RADIUS_M)
                if first:ds.to_zarr(path,mode='w-',consolidated=False,zarr_format=2);first=False
                else:ds.drop_vars('area').to_zarr(path,mode='a',append_dim='time',consolidated=False)
            FIP_INPUTS[name]=path
            register(path,'daily common-grid FIP inputs',[name])

In [ ]:
FIP_DATA={}
if RUN_FIP:
    if not FIP_GRID_AND_SUPPORT_VERIFIED or REGION_MASKS_PATH is None:
        raise ValueError('Physical grid/support and regional masks need verification')
    if not all(name in FIP_INPUTS for name in DIAGNOSTIC_CASES):
        raise ValueError('Configure certified daily masks for the selected controls, active closures and tensile cases')
    scores=[]
    with open_product(REGION_MASKS_PATH) as masks:
        masks=masks.load()
    for name in DIAGNOSTIC_CASES:
        path=FIP_INPUTS[name]
        if name not in EXPERIMENTS: raise ValueError('Out-of-scope experiment')
        with open_product(path) as inp:
            if inp.attrs.get('common_grid_verified') != 1:
                raise ValueError('Common physical grid not certified')
            if inp.area.attrs.get('units') not in ['m^2','m2']:
                raise ValueError('Comparison area must be m²')
            pair=wf.common_fip(inp.mod_mask,inp.obs_mask,OBS_START,OBS_END).compute()
            FIP_DATA[name]=pair
            for region in REGIONS:
                score=wf.spatial_scores(pair,inp.area,masks[region],threshold=.5)
                scores.append(dict(experiment=name,region=region,**score))
            path_out=EXPORT/f'FIP_{name}_{OBS_START}_{OBS_END}.nc'
            wf.save_cached(pair,path_out,SIGNATURE)
            register(path_out,'common-support FIP',[name],(OBS_START,OBS_END),'Fractions; mod minus obs')
    export_table(pd.DataFrame(scores),'FIP_spatial_skill',list(FIP_INPUTS))
    display(pd.DataFrame(scores))
else:
    print('Pending: certified common-support FIP inputs')

In [ ]:
if FIP_DATA:
    for label,(target,reference) in MAP_PAIRS.items():
        for region in REGIONS:
            fig,axes=plt.subplots(1,2,figsize=(11,4),sharex=True,sharey=True)
            for ax,name in zip(axes,[reference,target]):
                FIP_DATA[name]['diff'].where(masks[region]==1).plot(ax=ax,vmin=-1,vmax=1,cmap='RdBu_r',
                    cbar_kwargs={'label':'FIP model − AF2020 (fraction)'})
                ax.set_title(name)
            fig.suptitle(f'{label}, {region}: {OBS_START}–{OBS_END}');fig.tight_layout()
            path=EXPORT/f'{region}_{label}_FIPdiff_{OBS_START}_{OBS_END}.png'
            fig.savefig(path,dpi=220,bbox_inches='tight');plt.close(fig)
            register(path,'FIP comparison',[reference,target],notes='Projected diagnostic grid; geographic publication styling remains separate')

### Preparing and checking common-grid inputs

I'll verify the projection, longitude coordinates, remapping radius, source masks and validity before comparing FIP. A +0.25° shift is not accepted merely because a script defaults to it. Equal array indexes or period labels do not prove physical alignment. The optional recipe below points to the existing shuga remapping script; its independently averaged FIP outputs are exploratory unless their temporal denominators have been demonstrated equivalent.

The FIP calculation above deliberately requires reviewed daily masks on one physical grid. Record the source stores, remapping/interpolation method and validity policy in each input's attributes and set `common_grid_verified=1` only after that review. For native composites, choose and document the interval-to-daily representation first. Fractional concentration must not silently become binary occurrence. True zero persistence remains valid, while unobserved pixels remain missing. I'll keep support preparation and its recorded scientific choices here rather than treating a script output as automatically certified.

In [ ]:
recipe=['python3','scripts/comparisons/FIP_differencing.py','--experiment-group',GROUP,
        '--sim-name','Cs-high','--start-date',START,'--end-date',END,
        '--fip-start',OBS_START,'--fip-end',OBS_END,'--classification',METHOD,'--grid-type','Tc','--plot']
print('Exploratory remapping recipe, from shuga root:\n'+shlex.join(recipe))
print('Required reviewed input schema: mod_mask(time,y,x), obs_mask(time,y,x), area(y,x), common_grid_verified=1')

## 2. Mechanics, thickness and strength

I'll reconstruct staggered speed lazily on the full periodic grid, then crop before loading/reducing history, reconstruct speed on the T grid with shuga's Tc method, and check all units. Q90 of daily-history magnitudes is not a subdaily extreme. FI-conditioned means and persistence-weighted fields remain separate. Native strength units are retained; a vertically integrated N/m strength must not be labelled MPa. The `hi` convention determines whether division by concentration is needed before interpreting ice-covered-area thickness.

The main comparison set includes both matched controls, static/quadratic active closures and the static tensile sequence. I'll compare low speed with deformation and strength instead of interpreting immobility as proof of mechanical coherence. The code also counts valid exactly-zero velocities and reports whether classifications exclude them. Cache signatures include the analysis configuration, helper, source metadata and store modification times; replaced sources require a fresh reduction.

In [ ]:
MECHANICS={}
if RUN_MECHANICS:
    if HI_CONVENTION is None or NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED:
        raise ValueError('Review hi convention and exact T-grid coordinates first')
    from shuga.regridding.cice import compute_tgrid_speed
    with open_product(NATIVE_AREA_PATH) as native:native=native.load()
    wf.area_m2(native.tarea)
    zero_rows=[]
    for name in DIAGNOSTIC_CASES:
        history=load(name,'history',['hi','aice','divu','shear',STRENGTH_VARIABLE,'uvelE','uvelN','vvelE','vvelN'])
        classified=load(name,'mask',['FI_mask'])
        history,classified=wf.align_daily(history,classified,start=START,end=END)
        region=DIAGNOSTIC_REGION;bounds=ANTARCTIC_8_REGIONS[region]['plot_region']
        h=wf.crop_region(history,native.TLON,native.TLAT,bounds)
        mask=wf.crop_region(classified.FI_mask,native.TLON,native.TLAT,bounds)
        h,mask=xr.align(h,mask,join='exact')
        source=contexts(name)['pth_cfg'].resolve_cice_store()
        sig=wf.signature(dict(analysis=SIGNATURE,experiment=name,region=region,hi_convention=HI_CONVENTION,
            history_attrs=dict(history.attrs),mask_attrs=dict(classified.attrs),
            store=str(source),store_mtime=source.stat().st_mtime_ns))
        path=CACHE/f'{name}_{region}_mechanics_{START}_{END}.nc'
        if path.exists() and not RECOMPUTE_MECHANICS:
            with wf.read_cached(path,sig) as cached:MECHANICS[(name,region)]=cached.load()
            register(path,'mechanics cache',[name],(START,END),region);continue
        hi=wf.ice_thickness(h,HI_CONVENTION)
        fields=wf.fi_fields(hi,mask)
        fields['strain_q90']=wf.strain_q90(h)
        # Reconstruct lazily on the full periodic grid before cropping; avoid false tile-edge neighbours.
        speed_full=compute_tgrid_speed(history,history.aice,grid_type='Tc')
        speed=wf.crop_region(speed_full,native.TLON,native.TLAT,bounds)
        speed.attrs['units']='m/s'
        finite_ice=(h.aice>CFG['aice_thresh']) & np.isfinite(speed)
        zero=(speed==0)&finite_ice
        zero_rows.append(dict(experiment=name,region=region,valid_icy_samples=int(finite_ice.sum().compute()),
            exact_zero_samples=int(zero.sum().compute()),zero_samples_classified_FI=int((zero&(mask==1)).sum().compute())))
        fields['speed_mean']=speed.where(finite_ice).mean('time')
        strength=h[STRENGTH_VARIABLE]
        if not strength.attrs.get('units'):raise ValueError('Strength units must be explicit')
        sf=wf.fi_fields(strength,mask)
        fields['strength_conditional']=sf.hi_conditional
        fields['strength_weighted']=sf.hi_weighted
        fields=fields.compute();wf.save_cached(fields,path,sig)
        MECHANICS[(name,region)]=fields
        register(path,'mechanics cache',[name],(START,END),region)
    if zero_rows:export_table(pd.DataFrame(zero_rows),'exact_zero_speed_'+DIAGNOSTIC_REGION,DIAGNOSTIC_CASES,(START,END))

In [ ]:
if MECHANICS:
    for (name,region),ds in MECHANICS.items():
        fig,axes=plt.subplots(1,3,figsize=(14,4))
        for ax,variable in zip(axes,['strain_q90','hi_weighted','strength_weighted']):
            ds[variable].plot(ax=ax,cbar_kwargs={'label':ds[variable].attrs.get('units','')})
            ax.set_title(variable)
        fig.suptitle(f'{name}, {region}: {START}–{END}');fig.tight_layout()
        path=EXPORT/f'{name}_{region}_mechanics_{START}_{END}.png'
        fig.savefig(path,dpi=220,bbox_inches='tight');plt.close(fig)
        register(path,'mechanics figure',[name],(START,END),region)
    for label,(target,reference) in MAP_PAIRS.items():
        a=MECHANICS[(target,DIAGNOSTIC_REGION)];b=MECHANICS[(reference,DIAGNOSTIC_REGION)]
        a,b=xr.align(a,b,join='exact')
        diff=a-b;path=EXPORT/f'{label}_{DIAGNOSTIC_REGION}_mechanics_difference.nc'
        wf.save_cached(diff,path,SIGNATURE)
        register(path,'target minus reference mechanics',[target,reference],(START,END))

## 3. Actual lateral-drag stresses and activation

In the current develop source, `ice_dyn_shared.F90` describes Kux/Kuy as final face stresses, and `ice_history.F90` writes them in N/m². I'll verify that the archived executable and output definitions match that source before enabling this section. Old estimated `rho_i*hi*Ku` stresses are retired. Magnitudes are computed before time averaging at each collocated E or N face; a vector assembled from E-x and N-y components is not valid.

The code below exports face-specific stress magnitudes and temporal summaries. These are not a closed momentum budget. Relating them quantitatively to T-cell FI or to atmospheric/ocean stresses requires reviewed face-to-T mapping and collocated forcing diagnostics. No stress ratio is fabricated from incompatible locations. Phase-space work uses the reviewed T-grid speed/strain reductions; I'll document the sample and time support before treating it as evidence of a closure regime.

In [ ]:
STRESS_ROWS=[]
if RUN_STRESS:
    if not LD_STRESS_DEFINITION_VERIFIED:raise ValueError('Confirm executable provenance and actual stress definitions')
    if NATIVE_AREA_PATH is None or not NATIVE_GRID_VERIFIED:raise ValueError('Confirm native face/grid locations')
    with open_product(NATIVE_AREA_PATH) as native:native=native.load()
    for name in DIAGNOSTIC_CASES:
        h=load(name,'history',['KuxE','KuyE','KuxN','KuyN'])
        wf.daily_check(h,START,END)
        # The slice uses model indexes; labels remain face-specific, not T-grid stresses.
        tile=wf.crop_region(h,native.TLON,native.TLAT,ANTARCTIC_8_REGIONS[DIAGNOSTIC_REGION]['plot_region'])
        face=wf.ld_face_stress(tile)
        out=xr.Dataset()
        for v in face:
            series=face[v]
            if series.chunks is not None:series=series.chunk({'time':-1})
            out[v+'_mean']=series.mean('time');out[v+'_q90']=series.quantile(.9,'time')
            STRESS_ROWS.append(dict(experiment=name,face=v,maximum_daily_magnitude=float(series.max().compute()),units='Pa'))
        path=EXPORT/f'{name}_{DIAGNOSTIC_REGION}_face_stress_{START}_{END}.nc'
        wf.save_cached(out.compute(),path,SIGNATURE);register(path,'actual face stress',[name],(START,END))
    export_table(pd.DataFrame(STRESS_ROWS),'face_stress_summary',DIAGNOSTIC_CASES,(START,END))

## 4. Formation, retention, retreat and tendencies

I'll retain observation-date/composite definitions when interpreting onset. The first/last qualifying day and annual occurrence count below describe model classification, not an independently observed event date. Fragmented occupancy needs the daily record as context. Tendency sums inside a changing mask omit the mask-transition contribution and therefore cannot be called a closed FIA budget.

In [ ]:
if RUN_OCCURRENCE:
    rows=[]
    for name in DIAGNOSTIC_CASES:
        mask=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask)
        wf.daily_check(mask,START,END)
        for year,part in mask.groupby('time.year'):
            valid=part.notnull().sum('time');count=(part==1).sum('time')
            index=xr.DataArray(np.arange(part.sizes['time']),dims='time',coords={'time':part.time})
            first=index.where(part==1).min('time');last=index.where(part==1).max('time')
            out=xr.Dataset({'valid_days':valid,'FI_days':count,'first_index':first,'last_index':last})
            out.attrs.update(index_origin=str(pd.Timestamp(part.time.values[0]).date()),index_units='days from index_origin')
            path=EXPORT/f'{name}_occurrence_{year}.nc';wf.save_cached(out.compute(),path,SIGNATURE)
            register(path,'occurrence/retention',[name],(f'{year}-01-01',f'{year}-12-31'))
if RUN_TENDENCIES:
    if NATIVE_AREA_PATH is None or not MASK_TEMPORAL_SUPPORT_VERIFIED:raise ValueError('Review native area and tendency/mask timing')
    with open_product(NATIVE_AREA_PATH) as native:area=wf.area_m2(native.tarea).load()
    for name in DIAGNOSTIC_CASES:
        h=load(name,'history',['daidtt','daidtd'])
        mask=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask)
        h,mask=wf.align_daily(h,mask,start=START,end=END)
        result={}
        for v in ['daidtt','daidtd']:
            units=wf._unit(h[v])
            factor={'s-1':86400.,'1/s':86400.,'1/day':1.,'day-1':1.,'%/day':.01}.get(units)
            if factor is None:raise ValueError(f'Review {v} units: {units}')
            result[v]=(h[v]*factor*area).where(mask==1).sum(list(area.dims),min_count=1)/1e9
        ds=xr.Dataset(result);ds.attrs['interpretation']='FI-conditioned tendency sums; not a closed FIA budget'
        table=ds.compute().to_dataframe().reset_index();export_table(table,'FI_tendencies_'+name,[name],(START,END))

## 5. Geometry and distance from anchors

I'll use the executed FFx/FFy geometry file, not regenerate a different geometry in the analysis. Compare coastline-only, iceberg-only and combined cases with the same physics, forcing and dates. Distance below means distance to the nearest anchor-bearing **model-cell centre**; it is neither exact coastline distance nor an ocean-path distance. I'll check the form-factor coordinates against the native grid before enabling it.

In [ ]:
if RUN_ANCHOR:
    if not FORM_FACTOR_GRID_VERIFIED or FORM_FACTOR_PATH is None or NATIVE_AREA_PATH is None:
        raise ValueError('Review production form-factor identity and physical grid')
    with open_product(NATIVE_AREA_PATH) as native:native=native.load()
    with xr.open_dataset(FORM_FACTOR_PATH) as ff:
        anchor=((ff.FFx>0)|(ff.FFy>0)).astype(float).load()
    distance=wf.anchor_distance_km(native.TLON,native.TLAT,anchor)
    rows=[]
    for name in DIAGNOSTIC_CASES:
        mask=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask)
        wf.daily_check(mask,START,END)
        fip=mask.mean('time')
        for lower,upper in zip([0,10,25,50,100],[10,25,50,100,np.inf]):
            inside=((distance>=lower)&(distance<upper)).astype(float)
            value=wf.regional_means(fip,native.tarea,inside,xr.ones_like(inside))
            rows.append(dict(experiment=name,lower_km=lower,upper_km=upper,mean_fip=float(value.compute())))
    export_table(pd.DataFrame(rows),'anchor_distance',DIAGNOSTIC_CASES,(START,END))

## 6. Pack and total sea-ice checks

I'll use both a fixed geographical domain outside the anchoring zone and each run's changing pack classification. The fixed domain must be independent of the simulated outcome. I'll compare SIA, SIV/SIT, drift/speed and deformation without letting a changed mask masquerade as improved pack behaviour. Total volume depends on the reviewed hi convention; no extra concentration weighting is applied to a grid-cell volume-per-area field. Observational SIA skill is evaluated in the main notebook's common window; volume/thickness/drift remain model contrasts unless suitable observations are supplied.

In [ ]:
if RUN_PACK:
    if FIXED_PACK_MASK_PATH is None or NATIVE_AREA_PATH is None or HI_CONVENTION is None or not NATIVE_GRID_VERIFIED:
        raise ValueError('Review fixed pack domain, area, grid and hi convention')
    from shuga.regridding.cice import compute_tgrid_speed
    with open_product(NATIVE_AREA_PATH) as native:area=wf.area_m2(native.tarea).load()
    with open_product(FIXED_PACK_MASK_PATH) as ds:fixed=wf.binary_mask(ds.fixed_pack_mask).load()
    for name in DIAGNOSTIC_CASES:
        h=load(name,'history',['hi','aice','divu','shear','uvelE','uvelN','vvelE','vvelN'])
        fi=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask)
        h,fi=wf.align_daily(h,fi,start=START,end=END)
        ice=xr.where(h.aice.notnull(),(h.aice>.15).astype(float),np.nan)
        pack=xr.where(fi.notnull()&h.aice.notnull(),((h.aice>.15)&(fi==0)).astype(float),np.nan)
        hi=wf.ice_thickness(h,HI_CONVENTION)
        speed=compute_tgrid_speed(h,h.aice,grid_type='Tc')
        result=xr.Dataset({'fixed_hi':wf.regional_means(hi,area,fixed,ice),
            'pack_hi':wf.regional_means(hi,area,fixed,pack),
            'fixed_speed':wf.regional_means(speed,area,fixed,ice),
            'pack_speed':wf.regional_means(speed,area,fixed,pack)})
        for var in ['divu','shear']:
            result['fixed_'+var]=wf.regional_means(h[var],area,fixed,ice)
            result['pack_'+var]=wf.regional_means(h[var],area,fixed,pack)
        dims=list(area.dims)
        result['SIA']=(h.aice*area).sum(dims,min_count=1)/1e9
        result['SIV']=(hi*h.aice*area).sum(dims,min_count=1)/1e12
        result.SIA.attrs['units']='10^3 km^2';result.SIV.attrs['units']='10^3 km^3'
        export_table(result.compute().to_dataframe().reset_index(),'pack_'+name,[name],(START,END))

## 7. Point thickness context

I'll add a small point comparison only where dates, locations, thickness convention, snow and representativeness are known. BEPSII/core records sample local conditions and cannot be treated as a gridded fast-ice thickness climatology. The canonical input CSV below records a reviewed site sample; its provenance remains attached to the export. Exact model days are required. Nearest-cell offsets are reported, and no nearest-date substitution or unlabelled cross-era climatological comparison is made.

In [ ]:
if RUN_POINT_OBS:
    if not POINT_OBS_SAMPLING_VERIFIED or POINT_OBS_PATH is None or NATIVE_AREA_PATH is None or HI_CONVENTION is None:
        raise ValueError('Review observation sampling, provenance and thickness convention')
    from scipy.spatial import cKDTree
    points=pd.read_csv(POINT_OBS_PATH,parse_dates=['time'])
    required={'site','time','lon','lat','thickness_m'}
    if not required.issubset(points):raise ValueError('Point CSV schema is incomplete')
    with open_product(NATIVE_AREA_PATH) as native:native=native.load()
    lo=np.deg2rad(native.TLON.values);la=np.deg2rad(native.TLAT.values)
    xyz=np.stack([np.cos(la)*np.cos(lo),np.cos(la)*np.sin(lo),np.sin(la)],axis=-1)
    if 'ocean_mask' not in native:raise ValueError('Point matching requires reviewed binary ocean_mask on the native grid')
    good=np.isfinite(xyz).all(-1) & (wf.binary_mask(native.ocean_mask).values==1);flat=np.flatnonzero(good);tree=cKDTree(xyz[good])
    rows=[]
    for name in DIAGNOSTIC_CASES:
        h=load(name,'history',['hi','aice']);hi=wf.ice_thickness(h,HI_CONVENTION)
        fi=wf.binary_mask(load(name,'mask',['FI_mask']).FI_mask)
        hi,fi=wf.align_daily(hi,fi,start=START,end=END)
        for point in points.itertuples():
            day=pd.Timestamp(point.time).normalize()
            if day not in pd.DatetimeIndex(hi.time.values):raise ValueError(f'{point.site}: outside exact model-day support')
            qlo,qla=np.deg2rad([point.lon,point.lat]);q=[np.cos(qla)*np.cos(qlo),np.cos(qla)*np.sin(qlo),np.sin(qla)]
            chord,k=tree.query(q);ij=np.unravel_index(flat[k],native.TLON.shape)
            index=dict(zip(native.TLON.dims,ij))
            rows.append(dict(experiment=name,site=point.site,date=str(day.date()),
                observed_thickness_m=point.thickness_m,model_thickness_m=float(hi.sel(time=day).isel(index).compute()),
                model_FI=float(fi.sel(time=day).isel(index).compute()),
                grid_distance_km=2*6371*np.arcsin(np.clip(chord/2,0,1)),obs_source=str(POINT_OBS_PATH)))
    export_table(pd.DataFrame(rows),'point_thickness_context',DIAGNOSTIC_CASES,(START,END))

## 8. Export record and close sources

I'll use these exports to complete the spatial/mechanistic discussion and replace the development figures, with shared scales and clear conditioning. Publication map styling can reuse these verified numerical products. Missing observations or diagnostics remain limitations; a successful inventory run is not scientific validation. Return to the main notebook's review record after inspecting the outputs.

In [ ]:
(EXPORT/'supporting_artifacts.json').write_text(json.dumps(ARTIFACTS,indent=2))
(EXPORT/'supporting_review.json').write_text(json.dumps({
    'FIP_enabled':RUN_FIP,'FIP_support_reviewed':FIP_GRID_AND_SUPPORT_VERIFIED,
    'mechanics_enabled':RUN_MECHANICS,'hi_convention':HI_CONVENTION,
    'actual_stress_enabled':RUN_STRESS,'stress_definition_reviewed':LD_STRESS_DEFINITION_VERIFIED,
    'pack_enabled':RUN_PACK,'point_obs_enabled':RUN_POINT_OBS,
    'geometry_enabled':RUN_ANCHOR,'tendencies_enabled':RUN_TENDENCIES,
    'signature':SIGNATURE},indent=2))
for ds in OPEN.values():ds.close()
OPEN.clear()
print('Sources closed; disabled stages remain pending.')